# Qwen2.5-Coder-14B - IPDR fine-tune (optional) + Ollama deploy

Modified from your `qwen-2-5-14b-deployment` notebook.

**Flow**
1. Config (`RUN_FINETUNE`, dataset path, model tag)
2. *(Optional)* QLoRA on IPDR question→SQL JSONL → GGUF
3. Create Ollama model from fine-tuned GGUF **or** stock HF GGUF
4. Serve Ollama + Cloudflare tunnel (same as before)

Upload `finetune_ipdr.jsonl` (from `scripts/export_finetune_jsonl.py`) as a Kaggle dataset, then set `DATA_PATH` below.

> Fine-tune needs free GPU VRAM. Do **not** run Ollama serve during training. This notebook trains first (if enabled), then deploys.

**Update:** Install cell pins `datasets<5` and `trl<=1.15` (Unsloth-compatible). Restart kernel before re-running install.



## 0. Config

In [ ]:
# === Edit these ===
RUN_FINETUNE = True          # False = skip training, deploy stock Q6_K GGUF like before
DATA_PATH = "/kaggle/input/ipdr-finetune/finetune_ipdr.jsonl"  # adjust to your dataset path
MODEL_TAG = "qwen2.5-coder-14b-ipdr"   # Ollama name used by engines.yaml
STOCK_GGUF = "qwen2.5-coder-14b-instruct-q6_k.gguf"
MAX_SEQ_LEN = 2048
EPOCHS = 2
# If 14B QLoRA OOMs on your GPU, set USE_7B = True
USE_7B = False

print("RUN_FINETUNE =", RUN_FINETUNE)
print("DATA_PATH    =", DATA_PATH)
print("MODEL_TAG    =", MODEL_TAG)
print("USE_7B       =", USE_7B)

## 1. Machine check

In [ ]:
!uptime

In [ ]:
import subprocess
print(subprocess.getoutput(
    "nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv"
))

## 2. Optional: IPDR QLoRA fine-tune

Requires `DATA_PATH` JSONL with `messages` (system / user / assistant).  
Skip this section by setting `RUN_FINETUNE = False` in the config cell.

In [ ]:
import os
from pathlib import Path

FINETUNED_GGUF = Path("/kaggle/working/qwen-ipdr-gguf")
MERGED_DIR = Path("/kaggle/working/qwen-ipdr-merged")
ADAPTER_DIR = Path("/kaggle/working/outputs_ipdr_lora")

if not RUN_FINETUNE:
    print("Skipping fine-tune (RUN_FINETUNE=False).")
else:
    if not Path(DATA_PATH).exists():
        raise FileNotFoundError(
            f"Missing {DATA_PATH}. Upload finetune_ipdr.jsonl and fix DATA_PATH."
        )
    print("Dataset OK:", DATA_PATH, "bytes=", Path(DATA_PATH).stat().st_size)

In [ ]:
if RUN_FINETUNE:
    # Unsloth install for Kaggle. Internet ON. Fresh kernel. Do not import torch first.
    # Never run: pip install -U datasets trl  (pulls datasets 5.x / trl 1.19 — Unsloth rejects them).
    import os
    os.environ["WANDB_DISABLED"] = "true"
    %pip install -U pip
    %pip install -U unsloth
    # Force pins back if a previous cell upgraded them too far:
    %pip install --force-reinstall "datasets>=3.4.1,<5.0.0" "trl>=0.18.2,<=1.15.0,!=0.19.0"
    import importlib.metadata as md
    print("unsloth", md.version("unsloth"))
    print("datasets", md.version("datasets"), "(need <5)")
    print("trl", md.version("trl"), "(need <=1.15, not 0.19)")
else:
    print("skip install")



In [ ]:
if RUN_FINETUNE:
    from unsloth import FastLanguageModel
    import torch

    base = (
        "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
        if USE_7B
        else "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit"
    )
    print("Loading", base)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print("LoRA ready; GPU mem:")
    print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
else:
    model = tokenizer = None
    print("skip load")

In [ ]:
if RUN_FINETUNE:
    from datasets import load_dataset
    from trl import SFTTrainer
    from transformers import TrainingArguments
    import torch

    ds = load_dataset("json", data_files={"train": DATA_PATH}, split="train")

    def to_text(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}

    train_ds = ds.map(to_text, remove_columns=ds.column_names)
    print("train rows:", len(train_ds))

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            warmup_steps=20,
            num_train_epochs=EPOCHS,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            output_dir=str(ADAPTER_DIR),
            optim="adamw_8bit",
            seed=42,
            report_to="none",
        ),
    )
    trainer.train()
    print("training done")
else:
    print("skip train")

In [ ]:
if RUN_FINETUNE:
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    messages = [
        {
            "role": "system",
            "content": 'Write one SELECT for ipdr.ntc_ssl_log. Return only {"sql":"..."}.',
        },
        {
            "role": "user",
            "content": (
                "Question:\nTop 10 app labels by download bytes yesterday\n\n"
                "Label hints:\n(none)\n\nSimilar questions:\n(none)"
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    print(tokenizer.decode(out[0], skip_special_tokens=True)[-800:])
else:
    print("skip smoke test")

In [ ]:
if RUN_FINETUNE:
    # Merge adapters, then export GGUF for Ollama
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    FINETUNED_GGUF.mkdir(parents=True, exist_ok=True)
    print("Merging 16-bit weights ->", MERGED_DIR)
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print("Exporting Q4_K_M GGUF ->", FINETUNED_GGUF)
    try:
        model.save_pretrained_gguf(
            str(FINETUNED_GGUF),
            tokenizer,
            quantization_method="q4_k_m",
        )
    except Exception as exc:
        print("GGUF export via Unsloth failed:", exc)
        print("Merged HF folder is still at", MERGED_DIR)
        print("Convert offline with llama.cpp convert_hf_to_gguf.py + llama-quantize.")
        raise
    ggufs = sorted(FINETUNED_GGUF.rglob("*.gguf"))
    print("GGUF files:", ggufs)
    if not ggufs:
        raise FileNotFoundError("No .gguf produced under " + str(FINETUNED_GGUF))
else:
    print("skip merge/export")

In [ ]:
# Free training VRAM before Ollama loads the GGUF
if RUN_FINETUNE:
    import gc, torch
    try:
        del trainer
    except NameError:
        pass
    try:
        del model
        del tokenizer
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Freed training tensors.")
else:
    print("nothing to free")

## 3. Install Ollama + build Modelfile

Uses the fine-tuned GGUF when `RUN_FINETUNE=True`, otherwise downloads the stock Q6_K GGUF (your original behavior).

In [ ]:
!sudo apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

if RUN_FINETUNE:
    candidates = sorted(Path("/kaggle/working/qwen-ipdr-gguf").rglob("*.gguf"))
    if not candidates:
        raise FileNotFoundError("Fine-tune GGUF missing under /kaggle/working/qwen-ipdr-gguf")
    gguf_path = str(candidates[0])
    print("Using fine-tuned GGUF:", gguf_path)
else:
    gguf_path = hf_hub_download(
        repo_id="Qwen/Qwen2.5-Coder-14B-Instruct-GGUF",
        filename=STOCK_GGUF,
    )
    print("Using stock GGUF:", gguf_path)

# Keep SYSTEM as a single line so Ollama Modelfile parsing stays simple.
system_line = (
    "SYSTEM "
    + '"Write one SELECT for ipdr.ntc_ssl_log. '
    + 'Return only {\"sql\":\"...\"} or {\"clarify\":\"...\"}."'
)
modelfile = "\n".join(
    [
        f"FROM {gguf_path}",
        "PARAMETER num_ctx 8192",
        "PARAMETER temperature 0.1",
        "PARAMETER top_p 0.9",
        'PARAMETER stop "<|im_end|>"',
        'PARAMETER stop "<|endoftext|>"',
        system_line,
        "",
    ]
)
Path("Modelfile").write_text(modelfile, encoding="utf-8")
print("--- Modelfile ---")
print(modelfile)
print("MODEL_TAG =", MODEL_TAG)

## 4. Start Ollama and create the model

In [ ]:
import os, subprocess, time, requests

def running(name):
    out = subprocess.run(["pgrep", "-af", name], capture_output=True, text=True).stdout
    return any(name in line and "pgrep" not in line for line in out.splitlines())

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

if not running("ollama serve"):
    log = open("/tmp/ollama.log", "a")
    subprocess.Popen(
        ["env", "OLLAMA_KEEP_ALIVE=-1", "ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("listening")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read()[-2000:])

In [ ]:
!ollama --version

In [ ]:
import subprocess
cmd = ["ollama", "create", MODEL_TAG, "-f", "Modelfile"]
print("Running:", " ".join(cmd))
print(subprocess.getoutput(" ".join(cmd)))

In [ ]:
!ollama list

In [ ]:
import subprocess
print(subprocess.getoutput(
    "nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv"
))
print("---")
print(subprocess.getoutput("ollama ps"))
print("---")
print(open("/tmp/ollama.log", errors="replace").read()[-2000:])

In [ ]:
import requests

r = requests.post(
    "http://127.0.0.1:11434/v1/chat/completions",
    json={
        "model": f"{MODEL_TAG}:latest",
        "messages": [
            {
                "role": "system",
                "content": 'Write one SELECT for ipdr.ntc_ssl_log. Return only {"sql":"..."}.',
            },
            {
                "role": "user",
                "content": (
                    "Question:\nHow many flows started yesterday?\n\n"
                    "Label hints:\n(none)\n\nSimilar questions:\n(none)"
                ),
            },
        ],
        "temperature": 0.1,
    },
    timeout=300,
)
print("Status:", r.status_code)
print(r.text)

## 5. Cloudflare tunnel (public URL for engines.yaml)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!ollama ps

In [ ]:
!cloudflared --version

In [ ]:
import re
import subprocess
import time

def running(name):
    out = subprocess.run(["pgrep", "-af", name], capture_output=True, text=True).stdout
    return any(name in line and "pgrep" not in line for line in out.splitlines())

if not running("ollama serve"):
    subprocess.Popen(
        "nohup env OLLAMA_KEEP_ALIVE=-1 ollama serve > /tmp/ollama.log 2>&1",
        shell=True,
        start_new_session=True,
    )

open("/tmp/cloudflared.log", "a").close()
if not running("cloudflared"):
    subprocess.Popen(
        "nohup cloudflared tunnel --url http://127.0.0.1:11434 "
        "--http-host-header localhost:11434 >> /tmp/cloudflared.log 2>&1",
        shell=True,
        start_new_session=True,
    )

url = None
for _ in range(30):
    text = open("/tmp/cloudflared.log", errors="replace").read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break
    time.sleep(1)

print("Public URL:", url or "(not found yet)")
if not url:
    print(open("/tmp/cloudflared.log", errors="replace").read()[-2000:])
else:
    print()
    print("Put this in config.yaml -> kaggle.base_url")
    print("And engines.yaml model:", MODEL_TAG)
    print(f'  model: "{MODEL_TAG}"')
    print(f"  base_url: {url!r}  # or set via config.yaml kaggle.base_url")

In [ ]:
!tail -n 50 /tmp/ollama.log

## App wiring reminder

```yaml
# engines.yaml
- id: kaggle
  label: Kaggle 14B IPDR
  kind: ollama
  base_url: "https://YOUR-TUNNEL.trycloudflare.com"
  model: "qwen2.5-coder-14b-ipdr"
  num_ctx: 8192
  keep_alive: -1
```

Quick tunnels change every run — update `kaggle.base_url` each time.